# CNN Image Classification - Rock Paper Scissors
**Student ID:** 220121

This notebook trains a CNN using PyTorch on the Rock-Paper-Scissors dataset, then tests it on custom smartphone photos.

**Pipeline overview:**
1. Clone the GitHub repo to get custom phone images
2. Download and prepare the standard RPS training/test dataset
3. Define and train a CNN model
4. Evaluate on the test set with confusion matrix and error analysis
5. Run inference on 10 custom phone photos

## 1. Setup and Imports

In [ ]:
import os
import glob
import zipfile
import requests
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms, datasets

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Clone GitHub Repository (Custom Images)

In [ ]:
# clone my github repo to get the custom phone images
REPO_URL = 'https://github.com/Marwanthe0/CSE.git'
REPO_DIR = 'CSE'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print('Repository cloned successfully.')
else:
    print('Repository already exists, skipping clone.')

# path to custom images inside the cloned repo
CUSTOM_IMG_DIR = os.path.join(
    REPO_DIR,
    'Third Year', '3-2', 'Artificial Intelligence and ML', 'Lab Final', 'dataset'
)

# verify the custom images exist
if os.path.exists(CUSTOM_IMG_DIR):
    custom_files = [f for f in os.listdir(CUSTOM_IMG_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f'Found {len(custom_files)} custom images: {sorted(custom_files)}')
else:
    print(f'Warning: Custom image directory not found at {CUSTOM_IMG_DIR}')

## 3. Download Standard RPS Dataset

In [ ]:
# download URLs for the Rock-Paper-Scissors dataset by Laurence Moroney
TRAIN_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/rps.zip'
TEST_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip'

DATA_DIR = 'rps_data'
os.makedirs(DATA_DIR, exist_ok=True)

def download_and_extract(url, dest_dir):
    """Downloads a zip file from url and extracts it into dest_dir."""
    filename = url.split('/')[-1]
    filepath = os.path.join(dest_dir, filename)

    if not os.path.exists(filepath):
        print(f'Downloading {filename}...')
        response = requests.get(url, stream=True)
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f'Downloaded {filename}')
    else:
        print(f'{filename} already downloaded.')

    # extract
    with zipfile.ZipFile(filepath, 'r') as z:
        z.extractall(dest_dir)
    print(f'Extracted {filename}')

download_and_extract(TRAIN_URL, DATA_DIR)
download_and_extract(TEST_URL, DATA_DIR)

TRAIN_DIR = os.path.join(DATA_DIR, 'rps')
TEST_DIR = os.path.join(DATA_DIR, 'rps-test-set')

# show how many images per class
for split_name, split_dir in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    print(f'\n{split_name} set:')
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f'  {cls}: {count} images')

## 4. Data Preprocessing and DataLoaders

In [ ]:
IMG_SIZE = 150
BATCH_SIZE = 64

# training transforms with some augmentation to help the model generalize
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# for validation/test we only resize and normalize, no augmentation
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# load datasets using ImageFolder
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transform)

# class names and indices
class_names = train_dataset.classes
print(f'Classes: {class_names}')
print(f'Class to index mapping: {train_dataset.class_to_idx}')
print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

# create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### Quick look at some training samples

In [ ]:
# visualize a few training images
def show_batch(dataloader, class_names, n=8):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(1, n, figsize=(16, 3))
    for i in range(n):
        img = images[i].numpy().transpose((1, 2, 0))
        # undo normalization for display
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(class_names[labels[i]])
        axes[i].axis('off')
    plt.suptitle('Sample Training Images', fontsize=14)
    plt.tight_layout()
    plt.show()

show_batch(train_loader, class_names)

## 5. CNN Model Definition

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=3):
        super(CNN, self).__init__()

        # first conv block
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # second conv block
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # third conv block
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # classifier head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x

model = CNN(num_classes=len(class_names)).to(device)
print(model)

# count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 6. Training

In [ ]:
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# to store history for plotting
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
print('Starting training...\n')

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  |  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')

print('\nTraining complete!')

## 7. Save Model Weights

In [ ]:
# save the trained model state dict
MODEL_SAVE_DIR = 'saved_model'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_SAVE_DIR, '220121.pth')
torch.save(model.state_dict(), MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# also save to the cloned repo's model folder if it exists
repo_model_dir = os.path.join(
    REPO_DIR,
    'Third Year', '3-2', 'Artificial Intelligence and ML', 'Lab Final', 'model'
)
if os.path.exists(repo_model_dir):
    repo_model_path = os.path.join(repo_model_dir, '220121.pth')
    torch.save(model.state_dict(), repo_model_path)
    print(f'Also saved to {repo_model_path}')

## 8. Training History Plots

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# loss plot
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Training Loss', markersize=4)
ax1.plot(epochs_range, history['val_loss'], 'r-o', label='Validation Loss', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss vs Epochs')
ax1.legend()
ax1.grid(True, alpha=0.3)

# accuracy plot
ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Training Accuracy', markersize=4)
ax2.plot(epochs_range, history['val_acc'], 'r-o', label='Validation Accuracy', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Epochs')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 9. Confusion Matrix on Test Set

In [ ]:
# collect all predictions and true labels from the test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# print classification report
print('Classification Report:\n')
print(classification_report(all_labels, all_preds, target_names=class_names))

# confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix on Test Set')
plt.tight_layout()
plt.show()

## 10. Visual Error Analysis

In [ ]:
# find misclassified images from the test set
misclassified_indices = np.where(all_preds != all_labels)[0]
print(f'Total misclassified: {len(misclassified_indices)} out of {len(all_labels)}')

# pick 3 random misclassified samples (or fewer if less than 3 errors)
n_show = min(3, len(misclassified_indices))

if n_show > 0:
    chosen = random.sample(list(misclassified_indices), n_show)

    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
    if n_show == 1:
        axes = [axes]

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for i, idx in enumerate(chosen):
        img_tensor, true_label = test_dataset[idx]
        img = img_tensor.numpy().transpose((1, 2, 0))
        img = std * img + mean
        img = np.clip(img, 0, 1)

        pred_label = all_preds[idx]

        axes[i].imshow(img)
        axes[i].set_title(
            f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}',
            fontsize=12, color='red'
        )
        axes[i].axis('off')

    plt.suptitle('Misclassified Test Images', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No misclassified images found. The model got everything right!')

## 11. Custom Phone Image Predictions

In [ ]:
def predict_single_image(model, image_path, transform, class_names, device):
    """
    Loads an image, applies the transform, runs it through the model,
    and returns the predicted class name and confidence.
    """
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)
        confidence, pred_idx = torch.max(probs, 1)

    pred_class = class_names[pred_idx.item()]
    conf_pct = confidence.item() * 100
    return img, pred_class, conf_pct

In [ ]:
# get all custom images sorted by filename
custom_image_paths = sorted(glob.glob(os.path.join(CUSTOM_IMG_DIR, '*')))
custom_image_paths = [p for p in custom_image_paths
                      if p.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f'Found {len(custom_image_paths)} custom images to classify\n')

# run predictions on each image
results = []
for img_path in custom_image_paths:
    img, pred_class, conf = predict_single_image(
        model, img_path, test_transform, class_names, device
    )
    filename = os.path.basename(img_path)
    results.append((img, filename, pred_class, conf))
    print(f'{filename}  ->  Pred: {pred_class} ({conf:.1f}%)')

In [ ]:
# display all custom images in a 2x5 grid with predictions
n_images = len(results)
n_cols = 5
n_rows = (n_images + n_cols - 1) // n_cols  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten() if n_images > 1 else [axes]

for i, (img, filename, pred_class, conf) in enumerate(results):
    axes[i].imshow(img)
    # extract the true label from the filename (e.g. rock_1.jpg -> rock)
    true_label = filename.rsplit('_', 1)[0]
    color = 'green' if pred_class.lower() == true_label.lower() else 'red'
    axes[i].set_title(f'Pred: {pred_class} ({conf:.1f}%)\n({filename})',
                      fontsize=11, color=color, fontweight='bold')
    axes[i].axis('off')

# hide any extra subplot slots
for j in range(n_images, len(axes)):
    axes[j].axis('off')

plt.suptitle('Custom Phone Image Predictions', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 12. Summary

**What we did:**
- Trained a 3-block CNN from scratch on the standard Rock-Paper-Scissors dataset (2520 train, 372 test images)
- Used data augmentation (random flip, rotation, color jitter) to improve generalization
- The model architecture uses Conv2d -> BatchNorm -> ReLU -> MaxPool blocks followed by a fully connected classifier with dropout for regularization
- Evaluated the model with a confusion matrix, classification report, and visual error analysis
- Tested on 10 custom smartphone photos to see how well the model handles real world images